In [0]:
%sql
select count(1) from test_catalog.bronze.bronze_orders

In [0]:
%sql
-- Cost of the notebook query run at ~2026-07-14T01:22:27Z
-- Looks at all usage within a 5-min window around the query execution time
-- and attributes it to the identity that ran it
SELECT
  u.usage_start_time,
  u.usage_end_time,
  u.sku_name,
  u.usage_unit,
  u.usage_quantity,
  p.price,
  ROUND(u.usage_quantity * p.price, 6) AS cost,
  u.usage_metadata
FROM system.billing.usage u
JOIN (
  SELECT sku_name, pricing.effective_list.default AS price
  FROM system.billing.list_prices
  WHERE price_end_time IS NULL
) p ON u.sku_name = p.sku_name
WHERE
  u.usage_start_time >= '2026-07-14T01:20:00'
  AND u.usage_start_time <= '2026-07-14T01:25:00'
  AND u.identity_metadata.run_as = 'vasnamapparelandstyling@gmail.com'
ORDER BY u.usage_start_time

In [0]:
%sql
-- Cost breakdown per pipeline update run
-- Covers all compute consumed by the customer_orders pipeline
SELECT
  u.usage_metadata.dlt_update_id                      AS update_id,
  DATE(u.usage_start_time)                            AS run_date,
  MIN(u.usage_start_time)                             AS run_start,
  MAX(u.usage_end_time)                               AS run_end,
  u.sku_name,
  u.usage_unit,
  SUM(u.usage_quantity)                               AS total_dbu,
  MAX(p.price)                                        AS dbu_price,
  ROUND(SUM(u.usage_quantity) * MAX(p.price), 4)      AS total_cost_usd
FROM system.billing.usage u
JOIN (
  SELECT sku_name, pricing.effective_list.default AS price
  FROM system.billing.list_prices
  WHERE price_end_time IS NULL
) p ON u.sku_name = p.sku_name
WHERE
  u.usage_metadata.dlt_pipeline_id = 'cc56fabe-6fb6-4690-b39f-d5d9a916705d'
GROUP BY
  u.usage_metadata.dlt_update_id,
  DATE(u.usage_start_time),
  u.sku_name,
  u.usage_unit
ORDER BY
  run_date DESC,
  update_id,
  sku_name

In [0]:
%sql
select usage_metadata.notebook_id,usage_start_time,usage_end_time,usage_quantity,usage_unit,sku_name from system.billing.usage where usage_metadata.notebook_id is not null

In [0]:
%sql
-- Cost breakdown per pipeline update run
-- Covers all compute consumed by the customer_orders pipeline
SELECT
  u.usage_metadata.notebook_id,
  u.usage_metadata.dlt_update_id                      AS update_id,
  DATE(u.usage_start_time)                            AS run_date,
  MIN(u.usage_start_time)                             AS run_start,
  MAX(u.usage_end_time)                               AS run_end,
  u.sku_name,
  u.usage_unit,
  SUM(u.usage_quantity)                               AS total_dbu,
  MAX(p.price)                                        AS dbu_price,
  ROUND(SUM(u.usage_quantity) * MAX(p.price), 4)      AS total_cost_usd
FROM system.billing.usage u
JOIN (
  SELECT sku_name, pricing.effective_list.default AS price
  FROM system.billing.list_prices
  WHERE price_end_time IS NULL
) p ON u.sku_name = p.sku_name
WHERE
  u.usage_metadata.notebook_id is not null --in('1388900778354651','492994335035086')
GROUP BY
  u.usage_metadata.dlt_update_id,
  u.usage_metadata.notebook_id,
  DATE(u.usage_start_time),
  u.sku_name,
  u.usage_unit
ORDER BY
  run_date DESC,
  update_id,
  sku_name

In [0]:
df=spark.read.table('test_catalog.bronze.bronze_customer')
df.groupBy('customer_id').count().show()


In [0]:
spark.read.table('test_catalog.bronze.bronze_orders').groupBy('order_status').count().show()

In [0]:
%sql
SELECT DISTINCT _metadata.file_path
FROM test_catalog.bronze.bronze_customer;